# Micro-targeted vs General Audience Ads

This notebook builds a content-based classifier that uses `impressions_per_day` as a proxy label for audience breadth.

- Low impressions per day -> micro-targeted
- High impressions per day -> general audience
- Middle band is dropped to keep the labels clean

The analysis always keeps only the main parties used in `party_analysis.ipynb`.


In [ ]:
%load_ext watermark
%watermark -v -n -m -p numpy,pandas,scipy,sklearn,matplotlib

import os
import re
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

try:
    import seaborn as sns
except ImportError:
    sns = None

from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

%load_ext autoreload
%autoreload 2
%matplotlib inline

print(f'default sys.path: {sys.path}')
PROJ_ROOT = os.path.abspath(os.path.join(os.pardir))
PROJ_ROOT = os.path.abspath(os.path.join(PROJ_ROOT, os.pardir))
if PROJ_ROOT not in sys.path:
    sys.path.append(PROJ_ROOT)
print(f'Project root: {PROJ_ROOT}')

from utils import config
from utils import mpl_settings
from utils.dataset_utilities import load_data

mpl_settings.apply_settings()
plt.style.use(mpl_settings.plot_settings)

RANDOM_STATE = 42
MAIN_PARTIES = ['Liberal', 'Labor', 'Green', 'Independent', 'United Australia Party']


In [ ]:
file_path = '../../data/processed/2022_aus_elections_mar_to_may.csv'
df = load_data(file_path)
print(f'Raw shape: {df.shape}')

df = df[df['macro_party_uap'].isin(MAIN_PARTIES)].copy()
print(f'After main-party filter: {df.shape}')

display(df['macro_party_uap'].value_counts().rename('count').to_frame())


In [ ]:
analysis_df = df.copy()

required_columns = ['ad_delivery_start_time', 'ad_delivery_stop_time', 'mean_impressions', 'ad_creative_body', 'multiword_safe_lemmatized']
for col in required_columns:
    if col not in analysis_df.columns:
        analysis_df[col] = np.nan

analysis_df['start_time'] = pd.to_datetime(analysis_df['ad_delivery_start_time'], errors='coerce')
analysis_df['stop_time'] = pd.to_datetime(analysis_df['ad_delivery_stop_time'], errors='coerce')
analysis_df['stop_time'] = analysis_df['stop_time'].fillna(analysis_df['start_time'])

analysis_df = analysis_df[
    analysis_df['start_time'].notna()
    & analysis_df['stop_time'].notna()
    & analysis_df['mean_impressions'].notna()
].copy()

analysis_df['duration_days'] = (analysis_df['stop_time'] - analysis_df['start_time']).dt.days + 1
analysis_df['duration_days'] = analysis_df['duration_days'].clip(lower=1).astype(int)
analysis_df['impressions_per_day'] = analysis_df['mean_impressions'] / analysis_df['duration_days']
analysis_df = analysis_df[np.isfinite(analysis_df['impressions_per_day']) & (analysis_df['impressions_per_day'] > 0)].copy()

analysis_df['model_text'] = analysis_df['multiword_safe_lemmatized'].fillna(analysis_df['ad_creative_body']).fillna('').astype(str)
analysis_df['raw_text'] = analysis_df['ad_creative_body'].fillna(analysis_df['model_text']).fillna('').astype(str)
analysis_df = analysis_df[analysis_df['model_text'].str.len() > 0].copy()

# Use the low/high quantiles as a heuristic proxy for micro-targeted versus general-audience delivery.
low_q, high_q = 0.30, 0.70
low_thr = analysis_df['impressions_per_day'].quantile(low_q)
high_thr = analysis_df['impressions_per_day'].quantile(high_q)

analysis_df['audience_label'] = pd.NA
analysis_df.loc[analysis_df['impressions_per_day'] <= low_thr, 'audience_label'] = 'micro'
analysis_df.loc[analysis_df['impressions_per_day'] >= high_thr, 'audience_label'] = 'general'
analysis_df = analysis_df[analysis_df['audience_label'].notna()].copy()
analysis_df['target_general'] = (analysis_df['audience_label'] == 'general').astype(int)

print(f'Low threshold ({low_q:.0%} quantile): {low_thr:.2f}')
print(f'High threshold ({high_q:.0%} quantile): {high_thr:.2f}')
print(f'Final analysis shape: {analysis_df.shape}')

display(
    analysis_df[['impressions_per_day', 'duration_days', 'audience_label']].describe(include='all')
)
display(analysis_df['audience_label'].value_counts().rename('count').to_frame())


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

if sns is not None:
    sns.histplot(
        data=analysis_df,
        x='impressions_per_day',
        hue='audience_label',
        bins=40,
        element='step',
        stat='density',
        common_norm=False,
        ax=ax,
    )
else:
    micro = analysis_df.loc[analysis_df['audience_label'] == 'micro', 'impressions_per_day']
    general = analysis_df.loc[analysis_df['audience_label'] == 'general', 'impressions_per_day']
    ax.hist([micro, general], bins=40, density=True, label=['micro', 'general'], histtype='step')

ax.axvline(low_thr, color='firebrick', linestyle='--', linewidth=2, label=f'micro threshold: {low_thr:.1f}')
ax.axvline(high_thr, color='navy', linestyle='--', linewidth=2, label=f'general threshold: {high_thr:.1f}')
ax.set_xscale('log')
ax.set_xlabel('Impressions per day (log scale)')
ax.set_ylabel('Density')
ax.set_title('Label construction from impressions per day')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
def build_engineered_features(frame):
    text = frame['raw_text'].fillna('').astype(str)
    tokens = frame['model_text'].fillna('').astype(str).str.split()

    engineered = pd.DataFrame(index=frame.index)
    engineered['text_length'] = text.str.len()
    engineered['word_count'] = tokens.map(len)
    engineered['unique_word_ratio'] = tokens.map(lambda xs: len(set(xs)) / len(xs) if xs else 0.0)
    engineered['avg_word_length'] = tokens.map(lambda xs: float(np.mean([len(w) for w in xs])) if xs else 0.0)
    engineered['exclaim_count'] = text.str.count('!')
    engineered['question_count'] = text.str.count(r'\?')
    engineered['digit_count'] = text.str.count(r'\d')
    engineered['uppercase_ratio'] = text.map(lambda s: sum(ch.isupper() for ch in s) / max(len(s), 1))
    return engineered

feature_df = build_engineered_features(analysis_df)
analysis_df = pd.concat([analysis_df, feature_df], axis=1)

feature_cols = [
    'text_length',
    'word_count',
    'unique_word_ratio',
    'avg_word_length',
    'exclaim_count',
    'question_count',
    'digit_count',
    'uppercase_ratio',
]

X = analysis_df[['model_text'] + feature_cols].copy()
y = analysis_df['target_general'].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f'Training rows: {X_train.shape[0]}')
print(f'Test rows: {X_test.shape[0]}')
print('Train class balance:')
display(y_train.value_counts(normalize=True).rename('share').to_frame())


In [ ]:
text_transformer = TfidfVectorizer(
    min_df=5,
    max_df=0.9,
    ngram_range=(1, 2),
    stop_words='english',
    max_features=6000,
)

numeric_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler(with_mean=False)),
    ]
)

preprocess = ColumnTransformer(
    transformers=[
        ('text', text_transformer, 'model_text'),
        ('numeric', numeric_transformer, feature_cols),
    ],
    remainder='drop',
)

classifier = LogisticRegression(
    max_iter=2000,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    solver='liblinear',
)

model = Pipeline(
    steps=[
        ('preprocess', preprocess),
        ('classifier', classifier),
    ]
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

metrics = {
    'accuracy': accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred),
    'recall': recall_score(y_test, y_pred),
    'f1': f1_score(y_test, y_pred),
    'roc_auc': roc_auc_score(y_test, y_prob),
}

print(pd.Series(metrics).to_string())
print('\nClassification report (0 = micro, 1 = general):')
print(classification_report(y_test, y_pred, target_names=['micro', 'general']))


In [ ]:
cm = confusion_matrix(y_test, y_pred)
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = roc_auc_score(y_test, y_prob)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

if sns is not None:
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[0])
    axes[0].set_xticklabels(['micro', 'general'])
    axes[0].set_yticklabels(['micro', 'general'], rotation=0)
else:
    axes[0].imshow(cm, cmap='Blues')
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            axes[0].text(j, i, cm[i, j], ha='center', va='center')
    axes[0].set_xticks([0, 1])
    axes[0].set_yticks([0, 1])
    axes[0].set_xticklabels(['micro', 'general'])
    axes[0].set_yticklabels(['micro', 'general'])

axes[0].set_title('Confusion matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')

axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC AUC = {roc_auc:.3f}')
axes[1].plot([0, 1], [0, 1], color='gray', linestyle='--')
axes[1].set_xlabel('False positive rate')
axes[1].set_ylabel('True positive rate')
axes[1].set_title('ROC curve')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.show()


In [ ]:
feature_names = model.named_steps['preprocess'].get_feature_names_out()
coef = model.named_steps['classifier'].coef_.ravel()

coef_df = pd.DataFrame({'feature': feature_names, 'coef': coef})
coef_df['feature_display'] = coef_df['feature'].str.replace(r'^(text|numeric)__', '', regex=True)

top_general = coef_df.sort_values('coef', ascending=False).head(15).copy()
top_micro = coef_df.sort_values('coef', ascending=True).head(15).copy()

print('Most indicative features for general-audience ads (positive coefficients):')
display(top_general[['feature_display', 'coef']])

print('Most indicative features for micro-targeted ads (negative coefficients):')
display(top_micro[['feature_display', 'coef']])

fig, axes = plt.subplots(1, 2, figsize=(14, 8), sharex=False)
axes[0].barh(top_micro['feature_display'][::-1], top_micro['coef'][::-1], color='firebrick')
axes[0].axvline(0, color='black', linewidth=1)
axes[0].set_title('Micro-targeted cues')
axes[0].set_xlabel('Coefficient')

axes[1].barh(top_general['feature_display'][::-1], top_general['coef'][::-1], color='navy')
axes[1].axvline(0, color='black', linewidth=1)
axes[1].set_title('General-audience cues')
axes[1].set_xlabel('Coefficient')

plt.tight_layout()
plt.show()


## Propaganda Proxy Comparison

The next step uses a lightweight propaganda-style lexicon proxy. It is not a LIWC replacement; it is a small, transparent measure of mobilizing, fear-based, and urgency-heavy language.


In [ ]:
propaganda_lexicon = {
    'urgent',
    'urgently',
    'must',
    'now',
    'stop',
    'save',
    'protect',
    'fight',
    'danger',
    'dangerous',
    'crisis',
    'threat',
    'threaten',
    'corrupt',
    'corruption',
    'lie',
    'lies',
    'lying',
    'broken',
    'fail',
    'failed',
    'failure',
    'scam',
    'betray',
    'betrayal',
    'destroy',
    'disaster',
    'radical',
    'fear',
    'attack',
    'strong',
    'freedom',
}


def tokenize_for_lexicon(text):
    return re.findall(r"[a-z']+", str(text).lower())


def propaganda_score(text):
    tokens = tokenize_for_lexicon(text)
    if not tokens:
        return 0.0
    hits = sum(token in propaganda_lexicon for token in tokens)
    return 100.0 * hits / len(tokens)


analysis_df['propaganda_score'] = analysis_df['raw_text'].map(propaganda_score)
analysis_df['propaganda_hits'] = analysis_df['raw_text'].map(lambda t: sum(token in propaganda_lexicon for token in tokenize_for_lexicon(t)))

micro_scores = analysis_df.loc[analysis_df['audience_label'] == 'micro', 'propaganda_score']
general_scores = analysis_df.loc[analysis_df['audience_label'] == 'general', 'propaganda_score']

u_stat, p_val = stats.mannwhitneyu(micro_scores, general_scores, alternative='two-sided')
mean_micro = micro_scores.mean()
mean_general = general_scores.mean()
mean_diff = mean_micro - mean_general


def cohens_d(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    if len(a) < 2 or len(b) < 2:
        return np.nan
    pooled = np.sqrt(((len(a) - 1) * a.var(ddof=1) + (len(b) - 1) * b.var(ddof=1)) / (len(a) + len(b) - 2))
    if pooled == 0:
        return np.nan
    return (a.mean() - b.mean()) / pooled


d = cohens_d(micro_scores, general_scores)

print(f'Micro propaganda score mean:   {mean_micro:.3f}')
print(f'General propaganda score mean: {mean_general:.3f}')
print(f'Mean difference (micro - general): {mean_diff:.3f}')
print(f'Mann-Whitney U: {u_stat:.1f}')
print(f'p-value: {p_val:.4g}')
print(f"Cohen's d: {d:.3f}")

fig, ax = plt.subplots(figsize=(8, 5))

if sns is not None:
    sns.boxplot(data=analysis_df, x='audience_label', y='propaganda_score', order=['micro', 'general'], ax=ax)
    sns.stripplot(data=analysis_df, x='audience_label', y='propaganda_score', order=['micro', 'general'], color='0.25', alpha=0.20, size=2, ax=ax)
else:
    ax.boxplot([micro_scores, general_scores], labels=['micro', 'general'])

ax.set_title('Propaganda-style proxy by audience class')
ax.set_xlabel('Audience label')
ax.set_ylabel('Propaganda score per 100 words')
plt.tight_layout()
plt.show()

if p_val < 0.05:
    if mean_diff > 0:
        conclusion = 'Micro-targeted ads are more propagandistic on this proxy, and the difference is statistically significant.'
    else:
        conclusion = 'General-audience ads are more propagandistic on this proxy, and the difference is statistically significant.'
else:
    conclusion = 'The propaganda proxy does not show a statistically significant difference between micro-targeted and general-audience ads.'

print('\nConclusion:')
print(conclusion)


## Takeaway

If you want to extend this further, the next most useful addition would be to swap the proxy lexicon for a domain-specific dictionary and compare results by party or source page.
